# Movie Synopsis RoBERTa Embedding

시놉시스 텍스트만 한국어 Transformer 모델로 임베딩하여, 이후 MLP feature로 사용할 `.npy`와 index CSV를 생성합니다.
영화 제목은 사용하지 않습니다.

포스터 ResNet50 임베딩 노트북과 같은 방식으로 다음 산출물을 저장합니다.

```text
data/processed/synopsis_roberta/
  synopsis_embeddings_roberta_mean.npy
  synopsis_embeddings_roberta_mean_index.csv
  synopsis_embedding_log.csv
```

해당 데이터는 ResNet50 임베딩을 불러올 때처럼 index와 npy를 조합하여 사용가능합니다.

이 노트북은 시놉시스로 관객수 class를 예측하지 않습니다. 텍스트 의미 임베딩만 추출합니다.

## 1. 라이브러리 로드 및 드라이브 마운트

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    pass

print('torch:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. CSV 로드

경로 후보는 Colab Google Drive와 로컬 프로젝트 경로를 모두 지원합니다.

In [ ]:
csv_candidates = [
    Path('/content/gdrive/MyDrive/흥보위/sql_result/movie_snapshot_enriched_utf8_sig.csv'),
    Path('/content/drive/MyDrive/흥보위/sql_result/movie_snapshot_enriched_utf8_sig.csv'),
    Path('data/processed/movie_snapshot_enriched_utf8_sig.csv'),
    Path('../../data/processed/movie_snapshot_enriched_utf8_sig.csv'),
]

csv_file_path = next((path for path in csv_candidates if path.exists()), None)
if csv_file_path is None:
    raise FileNotFoundError('CSV 파일을 찾을 수 없습니다. csv_candidates 경로를 확인하세요.')

df = pd.read_csv(csv_file_path, encoding='utf-8-sig')

print(f'CSV 경로: {csv_file_path}')
print(f'전체 행 수: {len(df):,}')
print(df.columns.tolist())
df.head()

## 3. 2016년 이후 데이터 필터링

기존 모델링 실험과 맞추기 위해 기본값은 2016년 이후 영화만 대상으로 합니다.

In [ ]:
min_release_date = pd.Timestamp('2016-01-01')

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
model_df = df[df['release_date'] >= min_release_date].copy()

print(f'전체 행 수: {len(df):,}')
print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'제외된 행 수: {len(df) - len(model_df):,}')
model_df[['movie_name_clean', 'release_date', 'synopsis']].head()

## 4. 임베딩 대상 텍스트 구성

- `synopsis` 원문만 입력 텍스트로 사용합니다. 시놉시스가 비어 있는 영화는 기본적으로 제외합니다.
  - 시놉시스가 비어있는 영화는 모델 학습시 zerovector 등의 처리를 진행합니다.
- 영화 제목은 추후 필요시 별도로 추출하여 구성합니다.
  - 제목만 의미로 가지는 영화와 제목 + 줄거리의 의미를 가지는 영화간의 간극을 줄이기 위함입니다.

In [ ]:
required_columns = ['movie_name_clean', 'release_date', 'synopsis']
missing_columns = [col for col in required_columns if col not in model_df.columns]
if missing_columns:
    raise ValueError(f'필수 컬럼이 없습니다: {missing_columns}')

synopsis_text = model_df['synopsis'].fillna('').astype(str).str.strip()
valid_synopsis = synopsis_text != ''

synopsis_df = model_df[valid_synopsis].copy()
synopsis_df = synopsis_df.reset_index().rename(columns={'index': 'source_row_index'})

synopsis_df['embedding_text'] = synopsis_df['synopsis'].fillna('').astype(str).str.strip()

# 처음에는 100개 정도로 테스트하고, 전체 실행할 때 None으로 바꿉니다.
# MAX_ROWS = 100
MAX_ROWS = None
if MAX_ROWS is not None:
    synopsis_df = synopsis_df.head(MAX_ROWS).copy()

print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'시놉시스 있는 행 수: {valid_synopsis.sum():,}')
print(f'이번 실행 대상 행 수: {len(synopsis_df):,}')
synopsis_df[['source_row_index', 'movie_name_clean', 'release_date', 'embedding_text']].head()

## 5. Transformer 모델 로드

기본 모델은 `klue/roberta-large`입니다.

In [ ]:
MODEL_NAME = 'klue/roberta-large'
# MODEL_NAME = 'klue/roberta-base'

MAX_LENGTH = 512
BATCH_SIZE = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
text_model.eval()

hidden_size = text_model.config.hidden_size

print('모델:', MODEL_NAME)
print('hidden size:', hidden_size)
print('max length:', MAX_LENGTH)
print('batch size:', BATCH_SIZE)
print('device:', device)

## 6. Dataset/DataLoader 구성

In [ ]:
class SynopsisDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        return row['embedding_text'], index


def collate_synopsis_batch(batch):
    texts, indices = zip(*batch)
    tokenized = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    return tokenized, torch.tensor(indices, dtype=torch.long)


synopsis_dataset = SynopsisDataset(synopsis_df)
synopsis_loader = DataLoader(
    synopsis_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_synopsis_batch,
)

print('dataset size:', len(synopsis_dataset))

## 7. Mean Pooling 함수

padding 토큰을 제외하고 실제 토큰의 hidden state 평균을 냅니다. 긴 시놉시스에서는 `[CLS]` 하나보다 mean pooling이 더 안정적인 경우가 많습니다.

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

## 8. 시놉시스 임베딩 추출

In [ ]:
all_embeddings = []
all_indices = []
batch_logs = []

start_time = time.time()

with torch.no_grad():
    for batch_no, (tokenized, indices) in enumerate(tqdm(synopsis_loader), start=1):
        tokenized = {key: value.to(device) for key, value in tokenized.items()}
        outputs = text_model(**tokenized)
        embeddings = mean_pooling(outputs.last_hidden_state, tokenized['attention_mask'])

        all_embeddings.append(embeddings.cpu().numpy())
        all_indices.extend(indices.numpy().tolist())
        batch_logs.append({
            'batch_no': batch_no,
            'batch_size': len(indices),
            'max_token_length': int(tokenized['attention_mask'].sum(dim=1).max().item()),
        })

synopsis_embeddings = (
    np.vstack(all_embeddings).astype(np.float32)
    if all_embeddings
    else np.empty((0, hidden_size), dtype=np.float32)
)
embedding_index_df = synopsis_df.iloc[all_indices].reset_index(drop=True)
elapsed_sec = time.time() - start_time

print('embedding shape:', synopsis_embeddings.shape)
print('index rows:', len(embedding_index_df))
print('elapsed seconds:', round(elapsed_sec, 1))
embedding_index_df[['source_row_index', 'movie_name_clean', 'release_date']].head()

## 9. 임베딩 저장

In [ ]:
base_output_dir = csv_file_path.parent
output_base_dir = base_output_dir / 'synopsis_roberta'
output_base_dir.mkdir(parents=True, exist_ok=True)

embedding_npy_path = output_base_dir / 'synopsis_embeddings_roberta_mean.npy'
embedding_index_path = output_base_dir / 'synopsis_embeddings_roberta_mean_index.csv'
embedding_log_path = output_base_dir / 'synopsis_embedding_log.csv'

index_columns = [
    'source_row_index',
    'movie_name_clean',
    'release_date',
]

optional_columns = ['kmdb_doc_id']
index_columns += [col for col in optional_columns if col in embedding_index_df.columns]

embedding_index_save_df = embedding_index_df[index_columns].copy()
embedding_index_save_df['embedding_text_length'] = embedding_index_df['embedding_text'].astype(str).str.len()

embedding_log_df = pd.DataFrame(batch_logs)
embedding_log_df['model_name'] = MODEL_NAME
embedding_log_df['pooling'] = 'mean'
embedding_log_df['max_length'] = MAX_LENGTH
embedding_log_df['hidden_size'] = hidden_size
embedding_log_df['total_rows'] = len(embedding_index_save_df)

np.save(embedding_npy_path, synopsis_embeddings)
embedding_index_save_df.to_csv(embedding_index_path, index=False, encoding='utf-8-sig')
embedding_log_df.to_csv(embedding_log_path, index=False, encoding='utf-8-sig')

print(f'embedding 저장: {embedding_npy_path}')
print(f'index 저장: {embedding_index_path}')
print(f'log 저장: {embedding_log_path}')
print(f'index 컬럼: {embedding_index_save_df.columns.tolist()}')

## 10. 저장 결과 검증

In [ ]:
loaded_embeddings = np.load(embedding_npy_path)
loaded_index_df = pd.read_csv(embedding_index_path, encoding='utf-8-sig')

print('loaded embedding shape:', loaded_embeddings.shape)
print('loaded index rows:', len(loaded_index_df))
print('shape/index match:', loaded_embeddings.shape[0] == len(loaded_index_df))
print('embedding dim:', loaded_embeddings.shape[1] if loaded_embeddings.ndim == 2 else None)
print('index columns:', loaded_index_df.columns.tolist())

loaded_index_df.head()